In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# for GPU
from cuml.neighbors import KNeighborsClassifier
import cupy as cp 
from cuml.neighbors import KNeighborsRegressor

## Parameters for training

In [ ]:
# train_size = 4000
# train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv", nrows=train_size)
train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv")

conditions = [
    (train_data['booking_bool'] == 1),                     
    (train_data['click_bool'] == 1) & (train_data['booking_bool'] == 0)  
]
choices = [10, 5]
train_data['score'] = np.select(conditions, choices, default=1)

features = [
    'srch_length_of_stay',
    'srch_booking_window',
    'srch_adults_count',
    'srch_children_count',
    'srch_room_count',
    'srch_saturday_night_bool',
    'prop_id'
]

X = train_data[features]
y = train_data['score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_train = imputer.fit_transform(X_train)
X_train = scaler.fit_transform(X_train)
X_test = imputer.transform(X_test)
X_test = scaler.transform(X_test)

## Run with CPU: 

In [ ]:
k_values = [1, 3, 5, 11, 21, 51, 101, 201, 501, 2001]
all_accuracies = []
filtered_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    # accuracy on all test samples
    acc_all = accuracy_score(y_test, y_pred)
    all_accuracies.append(acc_all)
    
    # accuracy only for y_test == 5 or 10
    mask = (y_test == 5) | (y_test == 10)
    y_test_filtered = y_test[mask]
    y_pred_filtered = y_pred[mask]
    acc_filtered = accuracy_score(y_test_filtered, y_pred_filtered)
    filtered_accuracies.append(acc_filtered)
    
    print(f'k={k}, Test Accuracy (all): {acc_all:.4f}, Test Accuracy (only 5/10): {acc_filtered:.4f}')

plt.figure(figsize=(8, 5))
plt.plot(range(len(k_values)), all_accuracies, marker='o', label='All scores')
plt.plot(range(len(k_values)), filtered_accuracies, marker='s', label='Only score=5/10')
plt.xlabel('Number of Neighbors (k)', fontsize=14)
plt.ylabel('Test Accuracy', fontsize=14)
plt.title('k-NN Accuracy for Different k Values', fontsize=16)
plt.grid(True)
plt.xticks(ticks=range(len(k_values)), labels=k_values)
plt.legend(fontsize=12)
plt.show()

## Run with GPU: 

In [ ]:
X_train_gpu = cp.asarray(X_train)
X_test_gpu = cp.asarray(X_test)
y_train_gpu = cp.asarray(y_train)
y_test_gpu = cp.asarray(y_test)

k_values = [1, 3, 5, 11, 21, 51, 101, 201, 501, 2001]
all_accuracies = []
filtered_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_gpu, y_train_gpu)
    y_pred_gpu = knn.predict(X_test_gpu)
    y_pred = cp.asnumpy(y_pred_gpu)  

    # accuracy on all test samples
    acc_all = accuracy_score(y_test, y_pred)
    all_accuracies.append(acc_all)

    # accuracy only for y_test == 5 or 10
    mask = (y_test == 5) | (y_test == 10)
    y_test_filtered = y_test[mask]
    y_pred_filtered = y_pred[mask]
    acc_filtered = accuracy_score(y_test_filtered, y_pred_filtered)
    filtered_accuracies.append(acc_filtered)

    print(f'k={k}, Test Accuracy (all): {acc_all:.4f}, Test Accuracy (only 5/10): {acc_filtered:.4f}')


## Store the output from the previous cell

In [ ]:
k_values = [1, 3, 5, 11, 21, 51, 101, 201, 501, 2001]
all_accuracies = [0.9138, 0.952, 0.9549, 0.9552, 0.9552, 0.9552, 0.9552, 0.9552, 0.9552, 0.9552]
filtered_accuracies = [0.0262, 0.0028, 0.0003, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

## Plot

In [ ]:
plt.figure(figsize=(8, 5), dpi=300)
plt.plot(range(len(k_values)), all_accuracies, marker='o', label='All scores')
plt.plot(range(len(k_values)), filtered_accuracies, marker='s', label='Only score=5/10')

plt.xlabel('Number of Neighbors (k)', fontsize=14)
plt.ylabel('Test Accuracy', fontsize=14)
plt.title('k-NN Accuracy for Different k Values', fontsize=16)
plt.grid(True)
plt.xticks(ticks=range(len(k_values)), labels=k_values)
plt.legend(fontsize=12)

for i in range(len(k_values)):
    plt.text(i, all_accuracies[i] - 0.04, f"{all_accuracies[i]:.4f}",
             ha='center', va='top', fontsize=10, color='blue')
    plt.text(i, filtered_accuracies[i] + 0.04, f"{filtered_accuracies[i]:.4f}",
             ha='center', va='bottom', fontsize=10, color='red')

plt.savefig('KNN_results.png', dpi=300)
plt.show()

## Parameters for testing

In [ ]:
train_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/training_set_VU_DM.csv")
test_data = pd.read_csv("/kaggle/input/dmt-2025-2nd-assignment/test_set_VU_DM.csv")

conditions = [
    (train_data['booking_bool'] == 1),                     
    (train_data['click_bool'] == 1) & (train_data['booking_bool'] == 0)  
]
choices = [10, 5]
train_data['score'] = np.select(conditions, choices, default=1)

features = [
    'srch_length_of_stay',
    'srch_booking_window',
    'srch_adults_count',
    'srch_children_count',
    'srch_room_count',
    'srch_saturday_night_bool',
    'prop_id'
]

X_train = train_data[features]
y_train = train_data['score']

X_test = test_data[features]

imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()

X_train = imputer.fit_transform(X_train)
X_train = scaler.fit_transform(X_train)
X_test = imputer.transform(X_test)
X_test = scaler.transform(X_test)

## Test

### run with GPU: 

In [ ]:
def KNN_recommendation_gpu(k):
    knn_regressor = KNeighborsRegressor(n_neighbors=k)
    X_train_gpu = cp.asarray(X_train)
    y_train_gpu = cp.asarray(y_train)
    X_test_gpu = cp.asarray(X_test)
    
    knn_regressor.fit(X_train_gpu, y_train_gpu)
    predicted_score = knn_regressor.predict(X_test_gpu)

    test_data['predicted_score'] = cp.asnumpy(predicted_score)
    test_data_out = test_data[['srch_id', 'prop_id', 'predicted_score']]
    sorted_data = test_data_out.sort_values(
        by=['srch_id', 'predicted_score'],
        ascending=[True, False]
    ).reset_index(drop=True)
    sorted_data[['srch_id','prop_id']].to_csv(f"KNN_prediction_{k}_gpu.csv", index=False)

for k in [1,3,5]:
    KNN_recommendation_gpu(k)

### run with CPU: 

In [ ]:
def KNN_recommondation(k): 
    knn_regressor = KNeighborsRegressor(n_neighbors=k)
    knn_regressor.fit(X_train, y_train)
    
    test_data['predicted_score'] = knn_regressor.predict(X_test)
    test_data = test_data[['srch_id', 'prop_id', 'predicted_score']]
    sorted_data = test_data.sort_values(
        by=['srch_id', 'predicted_score'],
        ascending=[True, False]
    ).reset_index(drop=True)
    sorted_data[['srch_id','prop_id']].to_csv(f"KNN_prediction_{k}.csv", index=False)

for k in [1,3,5]: 
    KNN_recommondation(k)